# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aisyahnabillah/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = "month=2026-03"

dim_content = f"read_parquet('{REL}/dim_content.parquet')"
daily = f"read_parquet('{REL}/fact_content_daily_performance/{MONTH}/*.parquet')"

In [10]:
print(con.sql(f"DESCRIBE SELECT * FROM {daily} LIMIT 1").df()['column_name'].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [11]:
print(con.sql(f"DESCRIBE SELECT * FROM {dim_content} LIMIT 1").df()['column_name'].tolist())

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

One row in my analysis = one content item, on one day, for one client (grain: report_date + 
client_hash_id + content_hash_id), from fact_content_daily_performance.

I'm using one mid-panel month, month=2026-03, instead of the _sample table, because the lane 
guide warns the _sample table is the sealed final month (June 2026), not a random sample, and 
using it now would mean developing on the same window I'd later need as a clean test set.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(
    f"SELECT
        COUNT(*) AS row_count, 
        MIN(report_date) AS min_date, 
        MAX(report_date) AS max_date 
      FROM {daily}").show()

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Feature:** gsc_impressions, gsc_clicks, ctr, gsc_avg_position, word_count (from fact_content_daily), word_count and content_age_days (calculated from content_created_date in dim_content). All
known prior to or exactly at the report_date, making them safe to use.

**Label / proxy**: for this notebook's leakage demonstration only, I define a temporary proxy label, 
high_ctr = 1 if the CTR (calculated as gsc_clicks / gsc_impressions) exceeds that month's median.
This is not my lane's real target (Lane 1 is signal analysis, not classification), it exists only to reproduce the leakage trap on real warehouse data.

**Context:** client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, report_date, month are used for joining and grouping only, never fed to a model.

**Excluded:** I excluded sessions_ai and columns like ai_chatgpt/ai_perplexity/etc. from the main analysis because the lane guide indicates this AI-referral data is extremely sparse (30,177 rows out of 78.8 million in the entire warehouse) so treating them as standard features risks being misleading. I also excluded rows where is_deleted = TRUE, as well as GA4 data (ga4_sessions, etc.) for rows where ga4_data_available = FALSE, since those values ​​are zero-filled rather than being true zeros.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {daily}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").show()
# Expect zero rows back. If any come back, the grain claim in Section 1 is wrong.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
└─────────────┴────────────────┴─────────────────┴───────┘
                          0 rows                        



In [12]:
con.sql(f"""
    SELECT client_hash_id, COUNT(*) AS rows, MIN(report_date), MAX(report_date)
    FROM {daily}
    GROUP BY client_hash_id
    ORDER BY rows DESC
    LIMIT 10
""").show()

┌─────────────────────────┬────────┬──────────────────┬──────────────────┐
│     client_hash_id      │  rows  │ min(report_date) │ max(report_date) │
│         varchar         │ int64  │       date       │       date       │
├─────────────────────────┼────────┼──────────────────┼──────────────────┤
│ client_625b6439094e23e4 │ 988497 │ 2026-03-01       │ 2026-03-31       │
│ client_3ffa76342f366962 │ 904847 │ 2026-03-01       │ 2026-03-31       │
│ client_73cda7b4e4f265ea │ 869640 │ 2026-03-01       │ 2026-03-31       │
│ client_08a6a72ff48e62c0 │ 851275 │ 2026-03-01       │ 2026-03-31       │
│ client_62f4a7e64f5e0096 │ 756660 │ 2026-03-01       │ 2026-03-31       │
│ client_65de48885f4ef01b │ 426307 │ 2026-03-01       │ 2026-03-31       │
│ client_23a62021009f63c4 │ 423613 │ 2026-03-01       │ 2026-03-31       │
│ client_ba65e80a1116ae41 │ 410409 │ 2026-03-01       │ 2026-03-31       │
│ client_2b4306c3ed003f01 │ 375906 │ 2026-03-01       │ 2026-03-31       │
│ client_fef1a8f436438636

In [13]:
total = con.sql(f"SELECT COUNT(*) FROM {daily}").fetchone()[0]
available = con.sql(f"SELECT COUNT(*) FROM {daily} WHERE gsc_data_available IS TRUE").fetchone()[0]
print(f"Total rows: {total:,}")
print(f"Rows with gsc_data_available IS TRUE: {available:,} ({available/total:.1%})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 9,841,378
Rows with gsc_data_available IS TRUE: 3,611,061 (36.7%)


In [14]:
features_df = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        d.word_count,
        DATE_DIFF('day', d.content_created_date, f.report_date) AS content_age_days
    FROM {daily} f
    JOIN {dim_content} d ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_data_available IS TRUE AND f.gsc_impressions > 0 AND d.is_deleted IS FALSE
    LIMIT 5000
""").df()
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days
0,content_addcbcc385f558d4,2026-03-04,1,0,8.0,1022,191
1,content_ebd65e367786a5e9,2026-03-04,1,0,9.0,767,191
2,content_54119bbcc667c154,2026-03-04,1,0,33.0,899,191
3,content_2e0a49f1f81a65c1,2026-03-04,1,0,6.0,945,191
4,content_d1abccd2d15d2ab1,2026-03-04,1,0,9.0,881,191


1. gsc_impressions: knowable at the decision moment because it's an already-occurred impression count up to report_date, not a future number.
2. gsc_clicks: for the same reasoning, an already-occurred event on or before report_date.
3. gsc_avg_position: knowable because it's Search Console's recorded average position for that day, already observed, not predicted.
4. word_count: knowable because it's a property of the content itself, set when the content was created, not something dependent on future performance.
5. content_age_days: knowable because it's calculated from content_created_date, which has already happened relative to report_date.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = features_df.copy()
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]
median_ctr = df["ctr"].median()
df["high_ctr"] = (df["ctr"] > median_ctr).astype(int)

# Honest features. Deliberately NOT using raw gsc_clicks/gsc_impressions,
# since the label is computed directly from those two columns
X_honest = df[["gsc_avg_position", "word_count", "content_age_days"]].fillna(0)
y = df["high_ctr"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print(f"Honest AUC (no leak): {honest_auc:.3f}")

# Deliberate leak: ctr itself, the exact value the label was computed from
df["ctr_leak"] = df["ctr"] * 100
X_leaky = df[["gsc_avg_position", "word_count", "content_age_days", "ctr_leak"]].fillna(0)

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
leaky_auc = roc_auc_score(y_test2, model_leaky.predict_proba(X_test2)[:, 1])
print(f"Leaky AUC (with ctr_leak): {leaky_auc:.3f}  <- jumps toward 1.0, because the label was computed FROM this column")

print(f"\nFinal honest AUC kept for this notebook: {honest_auc:.3f}")

Honest AUC (no leak): 0.748
Leaky AUC (with ctr_leak): 0.999  <- jumps toward 1.0, because the label was computed FROM this column

Final honest AUC kept for this notebook: 0.748


: 

Adding ctr_leak, a column derived directly from the same ctr value the label was computed from, made the AUC jump toward 1.0, not because the model learned anything real, but because the feature and the label are circularly the same information. I removed ctr_leak and kept the honest AUC as the real number for this notebook. This is the same lesson as notebook 02's trend_pct/is_declining_label trap, reproduced here on real warehouse data instead of the starter CSV.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation:** this slice covers a single month (2026-03) for a subset of clients, and per the lane guide, client history depth is wildly unbalanced (some clients have 12+ months of history, others much less). A pattern found in this one month cannot be assumed to hold across seasons or across clients with very different history depth, and rows before a client's own ga4_data_start are GA4-zero-filled, not truly zero-engagement, so any GA4-based feature must be checked against ga4_data_available before being trusted.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.